# Lecture 2 (Part 2): Bond Valuation


## 2.0 Why bonds, and why a bank collapsed

The bond market is the largest securities market, larger than the stock market. We already know how to price them: a bond is nothing but a **stream of promised cash flows**.

Beyond pricing we will look into associated **risk**: *how much does the price move when interest rates move?* The answer to this question is conveyed through **bond duration**. In March 2023, Silicon Valley Bank, a 40-year-old institution with about \$200 billion in assets, collapsed in roughly 48 hours. The autopsy, at its core, is the derivative of the bond pricing formula with respect to $r$ — the exact formula we will discuss in this class, and at the end of class we will run SVB's trade ourselves.


---
## 2.1 Pricing a bond (vectorized)

A bond with payments $C_t$, $t = 1,\dots,N$ (the first $N{-}1$ are coupons; $C_N$ adds the principal), priced with a discount rate $r$:

$$P = \sum\limits_{t=1}^N \frac{C_t}{(1+r)^t}$$

We will assume a flat term structure of interest rates.

### Example: a two-year, 6% semi-annual coupon bond, discounted at 7%


You need to value a bond with face value of $100, a coupon rate of 6%, the coupons are paid semi-annually, and the current market rate is 7%. Is the price **(a)** above 100 **(b)** exactly 100 **(c)** below 100? *(Hint: the bond pays 6% when the market demands 7%. Is that bond attractive at a price of 100?)*

In [ ]:
import numpy as np

face_value = 100
coupon_rate = 0.06
semi_annual_coupon = coupon_rate / 2 * face_value

times = np.linspace(0.5, 2, 4)      # the four coupon dates -- linspace, not arange
r = 0.07

N = times.size
cash_flows = semi_annual_coupon * np.ones(N)
cash_flows[-1] += face_value        # a += b  means  a = a + b
print(times)
print(cash_flows)

In [ ]:
discounted_cash_flows = cash_flows / (1 + r/2)**(2*times)
bond_price = discounted_cash_flows.sum()
print(f"Bond Price is ${bond_price:.2f}")

<details>
<summary> <b>Explanation</b></summary>

Below 100, as expected: a bond paying less than the market demands must sell **at a discount**. A quick check: *coupon rate < market rate → price < face value; coupon rate > market rate → price > face value; equal → par.* Any bond price your code produces should pass this check before you believe anything else it says.

</details>



### Practice

Two bonds, both with €1,000 face value and 10 years remaining, market rate 7%:
- **Bond A**: just issued, coupon rate 7% (the current market rate).
- **Bond B**: issued 5 years ago at 13% coupons (rates were higher then).

**Note:** Bond A's price you can state *without computing* (why?). Bond B's — ballpark it. Then compute the price of Bond B, first with a loop, then vectorized.

In [ ]:
n_years_B = 10
face_value_B = 1000
coupon_rate_B = 0.13
interest_rate = 0.07

# non-numpy solution: a loop over the ten payment dates

print(f"The price of Bond B is {bond_price_B:.2f} Euro.")

In [ ]:
# numpy version: build the timeline, discount everything at once

print(f"The price of Bond B is {bond_price_B:.2f} Euro.")

<details>
<summary> <b>Solution</b></summary>

```python
n_years_B = 10
face_value_B = 1000
coupon_rate_B = 0.13
interest_rate = 0.07

# non-numpy solution: a loop over the ten payment dates
bond_price_B = face_value_B / (1 + interest_rate)**n_years_B
for t in range(1, n_years_B + 1):
    bond_price_B += coupon_rate_B * face_value_B / (1 + interest_rate)**t
print(f"The price of Bond B is {bond_price_B:.2f} Euro.")
```

```python
# numpy version: build the timeline, discount everything at once
times_B = np.arange(1, n_years_B + 1)                      # 1, 2, ..., 10
coupons_B = coupon_rate_B * face_value_B * np.ones(n_years_B)
coupons_B[-1] += face_value_B
bond_price_B = np.sum(coupons_B / (1 + interest_rate)**times_B)
print(f"The price of Bond B is {bond_price_B:.2f} Euro.")
```

Bond A trades at exactly €1,000 — par — because its coupon equals the market rate. Bond B pays 13% when the market offers 7%, so it must cost *more* than face value: about €1,421. Nobody is giving away 13% coupons at par.

</details>


---
## 2.2 Duration

**Macaulay duration** is the *time-weighted average* of when the bond's money actually arrives (weights = each payment's share of the price):

$$D = \frac{1}{P} \sum\limits_{t=1}^N \frac{tC_t}{(1+r)^t}$$

### Discussion

All three bonds below have 10-year maturity, market rate 7%. Rank their durations:

1. A **zero-coupon** bond (single payment at year 10)
2. Bond A (7% coupons)
3. Bond B (13% coupons)


In [ ]:
def duration_bond(F, c, r, n):
    times = np.arange(1, n + 1)
    coupons = c * F * np.ones(n)
    coupons[-1] += F
    P = np.sum(coupons / (1 + r)**times)
    return np.sum(times * coupons / (1 + r)**times) / P

interest_rate = 0.07
face_value = 1000
maturity = 10

duration_zero = duration_bond(face_value, 0.00, interest_rate, maturity)
duration_A    = duration_bond(face_value, 0.07, interest_rate, maturity)
duration_B    = duration_bond(face_value, 0.13, interest_rate, maturity)
print(f"Zero-coupon: {duration_zero:.2f} years")
print(f"Bond A (7%): {duration_A:.2f} years")
print(f"Bond B (13%): {duration_B:.2f} years")

The zero's duration is *exactly* its maturity — all the money arrives at year 10, so the "average arrival time" is 10. Coupons pull money earlier, so duration falls as coupons rise: B < A < zero. Same maturity, three different durations — **maturity is not interest-rate risk; duration is.**

### Duration as price sensitivity

Differentiate the pricing formula with respect to $r$:

$$\frac{dP}{dr} = \sum\limits_{t=1}^N \frac{-tC_t}{(1+r)^{t+1}} = - \frac{DP}{1+r}
\quad\Rightarrow\quad \frac{dP}{P} = -D\,\frac{dr}{1+r}$$

Duration is the **elasticity of the bond's price** with respect to $(1+r)$: a bond with duration 7 loses roughly 7% of its value when rates rise by one percentage point (from low levels). This is why duration is *the* risk number on every fixed-income desk.

### Discuss: is the risk symmetric?

Bond A, duration ≈ 7.5. Rates jump from 7% to 9% (+2pp), or drop to 5% (−2pp). The duration formula predicts roughly ∓14% either way. **Will the actual gain from −2pp be equal to the actual loss from +2pp?**

In [ ]:
def price_bond(F, c, r, n):
    times = np.arange(1, n + 1)
    coupons = F * c * np.ones(n)
    coupons[-1] += F
    return np.sum(coupons / (1 + r)**times)

face_value = 1000
coupon_rate = 0.07
maturity = 10

P0   = price_bond(face_value, coupon_rate, 0.07, maturity)
P_up = price_bond(face_value, coupon_rate, 0.09, maturity)     # rates +2pp
P_dn = price_bond(face_value, coupon_rate, 0.05, maturity)     # rates -2pp

print(f"Price at 7%: {P0:.2f}")
print(f"Rates +2pp:  {P_up:.2f}  ({(P_up/P0-1)*100:+.2f}%)")
print(f"Rates -2pp:  {P_dn:.2f}  ({(P_dn/P0-1)*100:+.2f}%)")

approx = duration_A * 0.02 / 1.07
print(f"Duration approximation: -/+ {approx*100:.2f}%")

<details>
<summary> <b>Explanation</b></summary>

The gain (+15.4%) is **bigger** than the loss (−12.8%), and the duration formula's ±14% splits the difference. The price–rate relationship is a *curve*, not a line — duration is only its slope at one point, a first-order approximation. The curvature is called **convexity**, and it works in the bondholder's favor: you gain more from falling rates than you lose from rising ones. For small rate moves duration is excellent; for 2008-sized or 2022-sized moves, mind the curve.

</details>



### Duration patterns

**Discuss**, then run: duration as a function of the coupon rate — increasing, decreasing, or flat?

In [ ]:
import matplotlib.pyplot as plt

F = 1000
n = 10
r = 0.07
coupon_rates = np.linspace(0, 0.17, 20)
durations = np.zeros(len(coupon_rates))
for i in range(len(coupon_rates)):
    durations[i] = duration_bond(F, coupon_rates[i], r, n)

plt.plot(coupon_rates, durations, 'b-', linewidth=2)
plt.ylabel("Duration"); plt.xlabel("Coupon rate"); plt.grid(True)

**The tricky one:** duration as a function of *maturity*, for a low-coupon bond discounted at a high rate (4% coupon, 15% market rate). "Longer maturity ⇒ higher duration" — always true?

In [ ]:
F = 1000
coupon_rate = 0.04
r = 0.15
maturities = np.arange(1, 61, 1)
durations = np.zeros(len(maturities))
for i in range(len(maturities)):
    durations[i] = duration_bond(F, coupon_rate, r, maturities[i])

plt.plot(maturities, durations, 'g-', linewidth=2)
plt.ylabel("Duration"); plt.xlabel("Maturity"); plt.grid(True)

Not always: at high discount rates and low coupons, the far-away principal is worth so little that *stretching the maturity further barely moves the average arrival time* — duration can even fall as maturity grows.

### Discussion

*"Duration is a proxy for the riskiness of a bond — so of two bonds, the riskier one should have the higher duration."* Do you agree? (Careful: what kinds of "risk" does a corporate bond have that a government bond doesn't — and what does a higher default risk do to promised coupon rates, and therefore to duration?)

---
## 2.3 The SVB trade

A *stylized* version of Silicon Valley Bank's balance sheet, 2021: using its high levels of deposits, the bank bought long-dated, safe government-backed bonds at the then-prevailing yields of about **1.6%**. Deposits are flighty (depositors can withdraw any day); the bonds pay back in about **10 years**. Then, in 2022-23, the Federal Reserve (FOMC) raised policy rates by nearly 5 percentage points — one of the fastest hiking cycles in history.

### Discuss

The bonds were *default-free* — the US government pays in full at maturity. So how much of the portfolio's market value could rising rates possibly destroy? **(a)** ~4% **(b)** ~10% **(c)** ~23% **(d)** default-free means no loss

In [ ]:
# stylized SVB portfolio: 10-year, 1.6% coupon bonds bought at par in 2021
portfolio = 91_000          # million dollars, roughly SVB's held-to-maturity book
r_2021 = 0.016
coupon = r_2021             # bought at par
r_2023 = 0.045
maturity = 10
face_value = 100

P_2021 = price_bond(face_value, coupon, r_2021, maturity)
P_2023 = price_bond(face_value, coupon, r_2023, maturity)  # same bonds, market rate now 4.5%

loss_pct = P_2023/P_2021 - 1
D = duration_bond(face_value, coupon, r_2021, maturity)

print(f"Duration of the portfolio at purchase: {D:.1f} years")
print(f"Price of the bonds after the rate rise: {P_2023:.2f} (bought at {P_2021:.2f})")
print(f"Mark-to-market loss: {loss_pct*100:.1f}%  =  ${abs(loss_pct)*portfolio:,.0f} million")

Roughly **−23%**, tens of billions of dollars — on *default-free* bonds. Duration of about 9 predicts a -27% loss on a ~3pp rate rise. The difference between predicted and actual loss is convexity.

The fatal ingredient was the *mismatch*: assets with duration ~9, funded by deposits with duration ~0 — money that can leave tonight. When depositors noticed the hole, they asked for **\$42 billion in a single day** (March 9, 2023), the bank had to sell bonds at these marked-down prices, and 48 hours later it was gone.

### Practice

One line each: (1) What would the loss have been if SVB had bought **3-year** bonds instead? (2) At what market rate would the 10-year portfolio lose exactly 10%? (Trial and error is fine — you have a pricing function.)


In [ ]:
# solution (1)


In [ ]:
# solution (2)


<details>
<summary> <b>Solution</b></summary>

```python
portfolio = 91_000          # million dollars, roughly SVB's held-to-maturity book
r_2021 = 0.016
coupon = r_2021             # bought at par
r_2023 = 0.045
maturity = 3
face_value = 100

P_2021 = price_bond(face_value, coupon, r_2021, maturity)          
P_2023 = price_bond(face_value, coupon, r_2023, maturity)          # same bonds, market rate now 4.5%
loss_pct = P_2023/P_2021 - 1
print(f"Mark-to-market loss: {loss_pct*100:.1f}%  =  ${abs(loss_pct)*portfolio:,.0f} million")
```

Maturity 3 years: about 8% loss.

```python
portfolio = 91_000          # million dollars, roughly SVB's held-to-maturity book
r_2021 = 0.016
coupon = r_2021             # bought at par
r_2023 = 0.0276
maturity = 10
face_value = 100

P_2021 = price_bond(face_value, coupon, r_2021, maturity)          
P_2023 = price_bond(face_value, coupon, r_2023, maturity)          # same bonds, market rate now 2.76%
loss_pct = P_2023/P_2021 - 1
print(f"Mark-to-market loss: {loss_pct*100:.1f}%  =  ${abs(loss_pct)*portfolio:,.0f} million")
```

At interest rate of 2.76% the portfolio loss is exactly 10%.

</details>

---
## 2.4 Bond Immunization Strategy: making a repayment promise rate-proof

A firm owes €1,000 (in present value) due in **10 years**, current rate 6%. It wants to buy bonds *today* such that, **whatever happens to interest rates**, the year-10 value of the bonds covers the obligation. A portfolio with this property is **immunized**.

Why is this hard? After a rate change, two forces fight each other: reinvested coupons earn the *new* rate (**reinvestment risk** — hurt by falling rates), while the bonds' resale price moves with rates (**price risk** — hurt by rising rates). Immunization is the art of making these two forces cancel.

The classical result: they cancel **when the portfolio's duration equals the obligation's horizon** (to a first-order approximation, and assuming a flat term structure that moves in parallel).

Three candidate bonds, all €1,000 face value:
- **Bond 1**: 10 years, 6.5% coupon
- **Bond 2**: 15 years, 7% coupon
- **Bond 3**: 30 years, 5.5% coupon

### Discuss

The obligation is due in 10 years. Which single bond immunizes it? *(The obvious answer is Bond 1 — it matures exactly in year 10. Is the obvious answer right? Recall: maturity is not interest-rate risk...)*

In [ ]:
r = 0.06
n = 10
PV = 1_000
FV = PV * (1 + r)**n
print(f"The future year-10 value of the obligation is {FV:.2f} Euro")

# the three bonds
n1 = 10
c1 = 0.065
F1 = 1_000

n2 = 15
c2 = 0.070
F2 = 1_000

#and a fancy way: pay attention to the order
n3, c3, F3 = 30, 0.055, 1_000

P_bond1 = price_bond(F1, c1, r, n1)
P_bond2 = price_bond(F2, c2, r, n2)
P_bond3 = price_bond(F3, c3, r, n3)
print(f"The bond prices are {P_bond1:.2f}, {P_bond2:.2f}, and {P_bond3:.2f} Euro")

# to invest exactly 1000 Euro in each bond, buy this many units:
w1, w2, w3 = PV/P_bond1, PV/P_bond2, PV/P_bond3
print(f"Units to purchase: {w1:.3f}, {w2:.3f}, and {w3:.3f}")

d1 = duration_bond(F1, c1, r, n1)
d2 = duration_bond(F2, c2, r, n2)
d3 = duration_bond(F3, c3, r, n3)
print(f"The bond durations are {d1:.2f}, {d2:.2f}, and {d3:.2f} years")

The durations are ≈ 7.7, **10.0**, and 14.8 years. The obligation's duration is 10 — and the bond that matches it is **Bond 2, the 15-year bond**, not the 10-year Bond 1. If you thought Bond 1 was the correct answer, so did quite a few real treasury departments. Coupons arrive early, pushing Bond 1's duration down to 7.7.

Let's look at the prices in year 10. In year 10, a bond position is worth: all coupons received so far, reinvested at the prevailing rate, plus the market price of the remaining bond. First at an unchanged 6%:

In [ ]:
coupon_FV_times = np.linspace(n-1, 0, n)     # years each coupon spends being reinvested

P10_bond1 = w1*(F1*c1*np.sum((1+r)**coupon_FV_times) + F1)
P10_bond2 = w2*(F2*c2*np.sum((1+r)**coupon_FV_times) + price_bond(F2, c2, r, n2-n))
P10_bond3 = w3*(F3*c3*np.sum((1+r)**coupon_FV_times) + price_bond(F3, c3, r, n3-n))
print(f"Year-10 values at unchanged rates: {P10_bond1:.2f}, {P10_bond2:.2f}, {P10_bond3:.2f}")
print(f"(the obligation: {FV:.2f})")

Immediately after purchase, the rate drops to 5% *and stays there*. For each bond: does its year-10 value rise or fall? Think through both channels — coupons now reinvest at only 5% (bad), but the year-10 resale price of any *remaining* maturity is higher (good). Which channel wins for the 10-year bond? For the 30-year?

In [ ]:
r_new = 0.05
P10_bond1_new = w1*(F1*c1*np.sum((1+r_new)**coupon_FV_times) + F1)
P10_bond2_new = w2*(F2*c2*np.sum((1+r_new)**coupon_FV_times) + price_bond(F2, c2, r_new, n2-n))
P10_bond3_new = w3*(F3*c3*np.sum((1+r_new)**coupon_FV_times) + price_bond(F3, c3, r_new, n3-n))
print(f"Year-10 values if rates fall to 5%: {P10_bond1_new:.2f}, {P10_bond2_new:.2f}, {P10_bond3_new:.2f}")
print(f"(the obligation: {FV:.2f})")

Bond 1 now **underfunds** the obligation (reinvestment risk won — it has no price risk left in year 10), Bond 3 **overfunds** it (price risk won), and Bond 2 barely moves. Now the full picture, for every rate at once:

In [ ]:
r_all = np.linspace(0, 0.15, 16)
p1 = np.zeros(len(r_all)); p2 = np.zeros(len(r_all)); p3 = np.zeros(len(r_all))

for i in range(len(r_all)):
    p1[i] = w1*(F1*c1*np.sum((1+r_all[i])**coupon_FV_times) + F1)
    p2[i] = w2*(F2*c2*np.sum((1+r_all[i])**coupon_FV_times) + price_bond(F2, c2, r_all[i], n2-n))
    p3[i] = w3*(F3*c3*np.sum((1+r_all[i])**coupon_FV_times) + price_bond(F3, c3, r_all[i], n3-n))

plt.plot(r_all, p1, '-d', r_all, p2, '-o', r_all, p3, '-s', linewidth=2)
plt.axhline(FV, color='k', linestyle=':')
plt.xlabel('Interest rate'); plt.ylabel('Year-10 Value of Bond Positions')
plt.legend(['Bond 1 (D=7.7)', 'Bond 2 (D=10.0)', 'Bond 3 (D=14.8)', 'Obligation'])
plt.ylim([1000, 3000])

Takeaways: to immunize, buy **Bond 2** — match *duration*, not maturity. And the match is only first-order: for large rate moves the curves drift apart (there is the convexity from Section 2.2 again — notice Bond 2's curve never dips *below* the obligation line).

### Linearity of duration

Portfolio duration is the value-weighted average of its bonds' durations. So a *mix* of Bond 1 ($D=7.7$) and Bond 3 ($D=14.8$) can also hit 10:

$$w_{bond_1} \, D_1 + (1-w_{bond_1})\, D_3 = 10$$

In [ ]:
w_bond1 = (d3 - n) / (d3 - d1)
print(f"Invest in Bond 1: {1000*w_bond1:.0f} Euro, in Bond 3: {1000*(1-w_bond1):.0f} Euro")

### Practice

1. Compute the year-10 value of this two-bond portfolio if the rate drops to 5% immediately after purchase. Does it cover the obligation?
2. Plot the portfolio's year-10 value against the interest rate, next to Bond 2's. Which immunizer is better far from 6% — the single duration-matched bond, or the *barbell* of a short and a long bond? One sentence on why. *(Hint: which portfolio has more convexity — more curvature working in your favor?)*



In [ ]:
# solution


<details>
<summary> <b>Solution</b></summary>

```python
# 1.
P10_ptf_new = w_bond1*P10_bond1_new + (1-w_bond1)*P10_bond3_new
print(f"Year-10 value of bond portfolio at 5%: {P10_ptf_new:.2f} Euro")

# 2.
p_ptf = w_bond1*p1 + (1-w_bond1)*p3
plt.plot(r_all, p2, '-d', r_all, p_ptf, '-s', linewidth=2)
plt.axhline(FV, color='k', linestyle=':')
plt.xlabel('Interest rate'); plt.ylabel('Year-10 Value')
plt.legend(['Bond 2', 'Barbell portfolio (Bonds 1+3)'])
plt.ylim([1000, 3000])
```

The barbell stays *above* the obligation by more, in both directions: same duration, higher convexity — spread-out cash flows curve harder. (In practice the market knows this and makes you pay for convexity, but that story belongs to a fixed-income course.)

</details>